# Practical 4 – MovieLens 25M (RAM-efficient version)

Run `Runtime → Restart session` before starting. Training/test split files are already on Drive.

Ratings are stored as CSR matrices (`int32` indices + `float32` values, 8 bytes per rating, ≈ 360 MB total). The ALS solver is implemented in numba (parallel, no temporary arrays).

In [ ]:
import gc, os, time
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.sparse import csr_matrix
from numba import njit, prange
from google.colab import drive
drive.mount('/content/drive')

FOLDER      = '/content/drive/MyDrive/ml-25m'
movies_file = f'{FOLDER}/movies.csv'
train_file  = f'{FOLDER}/4train_ratings_90.csv'
test_file   = f'{FOLDER}/4test_ratings_10.csv'
os.makedirs("figures", exist_ok=True)

## 1. Train / test split
The existing split uses 90% of each user's ratings for training and 10% for testing. Every user (≥ 20 ratings) has ratings in both sets.

In [ ]:
movies   = pd.read_csv(movies_file, usecols=["movieId", "title"])
item_ids = movies.movieId.values.astype(np.int64)
titles   = movies.title.values
n_item   = len(item_ids)
item_map = np.full(item_ids.max() + 1, -1, dtype=np.int32)
item_map[item_ids] = np.arange(n_item, dtype=np.int32)
N_USER   = 162541

def load_csr(path, chunksize=4_000_000):
    us, its, rs = [], [], []
    for ch in pd.read_csv(path, usecols=["userId", "movieId", "rating"],
                          dtype={"userId": np.int32, "movieId": np.int32, "rating": np.float32},
                          chunksize=chunksize):
        us.append(ch.userId.values - 1)
        its.append(item_map[ch.movieId.values])
        rs.append(ch.rating.values)
    u, i, r = np.concatenate(us), np.concatenate(its), np.concatenate(rs)
    del us, its, rs
    assert u.max() < N_USER and i.min() >= 0
    return csr_matrix((r, (u, i)), shape=(N_USER, n_item))

Ru  = load_csr(train_file)
Ri  = Ru.T.tocsr()
Rte = load_csr(test_file)
gc.collect()

mu = float(Ru.data.mean(dtype=np.float64))
item_cnt_train = np.diff(Ri.indptr)
print(f"train {Ru.nnz:,}  test {Rte.nnz:,}  mu {mu:.4f}")
print(f"CSR memory: {(Ru.data.nbytes+Ru.indices.nbytes+Ri.data.nbytes+Ri.indices.nbytes+Rte.data.nbytes+Rte.indices.nbytes)/1e6:.0f} MB")

## 2. Parallel ALS (numba)
Model: `r ≈ μ + b_u + b_i + U_u · V_i`, with four block updates (user biases, item biases, U, V). The same kernel serves both sides of the model.

In [ ]:
@njit
def chol_solve(A, b, k):
    for i in range(k):
        for j in range(i + 1):
            s = A[i, j]
            for p in range(j):
                s -= A[i, p] * A[j, p]
            if i == j:
                A[i, i] = np.sqrt(s)
            else:
                A[i, j] = s / A[j, j]
    y = np.empty(k)
    for i in range(k):
        s = b[i]
        for p in range(i):
            s -= A[i, p] * y[p]
        y[i] = s / A[i, i]
    x = np.empty(k)
    for i in range(k - 1, -1, -1):
        s = y[i]
        for p in range(i + 1, k):
            s -= A[p, i] * x[p]
        x[i] = s / A[i, i]
    return x

@njit(parallel=True, fastmath=True)
def update_bias(indptr, indices, data, F_row, F_col, b_col, mu, lam, gam, b_row):
    k = F_row.shape[1]
    for m in prange(len(indptr) - 1):
        a = indptr[m]
        e = indptr[m + 1]
        if e > a:
            s = 0.0
            for p in range(a, e):
                j = indices[p]
                d = 0.0
                for f in range(k):
                    d += F_row[m, f] * F_col[j, f]
                s += data[p] - mu - b_col[j] - d
            b_row[m] = lam * s / (lam * (e - a) + gam)

@njit(parallel=True, fastmath=True)
def update_factors(indptr, indices, data, b_row, b_col, mu, F_col, lam, tau, F_row):
    k = F_row.shape[1]
    for m in prange(len(indptr) - 1):
        a = indptr[m]
        e = indptr[m + 1]
        if e > a:
            A = np.zeros((k, k))
            rhs = np.zeros(k)
            for x in range(k):
                A[x, x] = tau
            for p in range(a, e):
                j = indices[p]
                res = data[p] - mu - b_row[m] - b_col[j]
                for x in range(k):
                    w = lam * F_col[j, x]
                    rhs[x] += w * res
                    for y in range(x + 1):
                        A[x, y] += w * F_col[j, y]
            F_row[m, :] = chol_solve(A, rhs, k)

@njit(parallel=True, fastmath=True)
def sse_rows(indptr, indices, data, F_row, F_col, b_row, b_col, mu):
    k = F_row.shape[1]
    out = np.zeros(len(indptr) - 1)
    for m in prange(len(indptr) - 1):
        s = 0.0
        for p in range(indptr[m], indptr[m + 1]):
            j = indices[p]
            d = 0.0
            for f in range(k):
                d += F_row[m, f] * F_col[j, f]
            err = data[p] - mu - b_row[m] - b_col[j] - d
            s += err * err
        out[m] = s
    return out

@njit(parallel=True)
def row_std(indptr, data):
    out = np.zeros(len(indptr) - 1)
    for m in prange(len(indptr) - 1):
        c = indptr[m + 1] - indptr[m]
        if c > 1:
            s = 0.0
            s2 = 0.0
            for p in range(indptr[m], indptr[m + 1]):
                x = data[p]
                s += x
                s2 += x * x
            out[m] = np.sqrt(max(s2 / c - (s / c) ** 2, 0.0))
    return out

def als(Ru, Ri, Rte, k, lam, gam, tau, iters, mu, seed=0, verbose=True):
    n_user, n_item = Ru.shape
    rng = np.random.default_rng(seed)
    U = 0.1 * rng.standard_normal((n_user, k))
    V = 0.1 * rng.standard_normal((n_item, k))
    U[np.diff(Ru.indptr) == 0] = 0.0
    V[np.diff(Ri.indptr) == 0] = 0.0
    bu = np.zeros(n_user); bi = np.zeros(n_item)
    N = Ru.nnz
    N_te = Rte.nnz if Rte is not None else 0
    hist = {"train": [], "test": [], "loss": []}
    t0 = time.time()
    for it in range(iters):
        update_bias(Ru.indptr, Ru.indices, Ru.data, U, V, bi, mu, lam, gam, bu)
        update_bias(Ri.indptr, Ri.indices, Ri.data, V, U, bu, mu, lam, gam, bi)
        update_factors(Ru.indptr, Ru.indices, Ru.data, bu, bi, mu, V, lam, tau, U)
        update_factors(Ri.indptr, Ri.indices, Ri.data, bi, bu, mu, U, lam, tau, V)
        sse_tr = sse_rows(Ru.indptr, Ru.indices, Ru.data, U, V, bu, bi, mu)
        hist["train"].append(np.sqrt(sse_tr.sum() / N))
        nb, nf = n_user + n_item, n_user + n_item
        hist["loss"].append(
            lam / 2 * sse_tr.sum() - N / 2 * np.log(lam / (2 * np.pi))
            - nb / 2 * np.log(gam / (2 * np.pi)) + gam / 2 * (bu @ bu + bi @ bi)
            - nf * k / 2 * np.log(tau / (2 * np.pi)) + tau / 2 * ((U * U).sum() + (V * V).sum()))
        sse_te = None
        if Rte is not None:
            sse_te = sse_rows(Rte.indptr, Rte.indices, Rte.data, U, V, bu, bi, mu)
            hist["test"].append(np.sqrt(sse_te.sum() / N_te))
        if verbose:
            te = f"{hist['test'][-1]:.4f}" if Rte is not None else "-"
            print(f"  it {it+1:2d}  train {hist['train'][-1]:.4f}  test {te}  ({time.time()-t0:.0f}s)")
    return dict(U=U, V=V, bu=bu, bi=bi, mu=mu, k=k, lam=lam, gam=gam, tau=tau,
                hist=hist, sse_tr=sse_tr, sse_te=sse_te)

### 2.1 Numerical equivalence and speed-up
Compare the numba kernel against a pure Python reference on a random subset of 5,000 users, then compare wall-clock times.

In [ ]:
print("os.cpu_count() =", os.cpu_count())

rng = np.random.default_rng(0)
n_sub = 5_000
idx = rng.choice(Ru.shape[0], size=n_sub, replace=False)
idx.sort()
Ru_sub = Ru[idx]
Ri_sub = Ru_sub.T.tocsr()

def update_factors_naive(indptr, indices, data, b_row, b_col, mu, F_col, lam, tau, F_row):
    k = F_row.shape[1]
    for m in range(len(indptr) - 1):
        a, e = indptr[m], indptr[m + 1]
        if e > a:
            A = tau * np.eye(k)
            rhs = np.zeros(k)
            for p in range(a, e):
                j = indices[p]
                res = data[p] - mu - b_row[m] - b_col[j]
                A += lam * np.outer(F_col[j], F_col[j])
                rhs += lam * F_col[j] * res
            F_row[m] = np.linalg.solve(A, rhs)

k, lam, tau, gam = 10, 1.0, 10.0, 0.1
rng = np.random.default_rng(0)
U1 = 0.1 * rng.standard_normal((Ru_sub.shape[0], k))
V1 = 0.1 * rng.standard_normal((Ru_sub.shape[1], k))
U2, V2 = U1.copy(), V1.copy()
bu = np.zeros(Ru_sub.shape[0]); bi = np.zeros(Ru_sub.shape[1])
mu_sub = float(Ru_sub.data.mean(dtype=np.float64))

update_factors(Ru_sub.indptr, Ru_sub.indices, Ru_sub.data, bu, bi, mu_sub, V1, lam, tau, U1)
update_factors_naive(Ru_sub.indptr, Ru_sub.indices, Ru_sub.data, bu, bi, mu_sub, V2, lam, tau, U2)
print("max |U_numba - U_naive| =", np.abs(U1 - U2).max())

t0 = time.time()
for _ in range(3):
    update_factors(Ru_sub.indptr, Ru_sub.indices, Ru_sub.data, bu, bi, mu_sub, V1, lam, tau, U1)
t_numba = (time.time() - t0) / 3

t0 = time.time()
for _ in range(3):
    update_factors_naive(Ru_sub.indptr, Ru_sub.indices, Ru_sub.data, bu, bi, mu_sub, V2, lam, tau, U2)
t_naive = (time.time() - t0) / 3

print(f"numba : {t_numba:.3f} s per sweep (5,000 users)")
print(f"naive : {t_naive:.3f} s per sweep (5,000 users)")
print(f"measured speed-up: {t_naive/t_numba:.1f}x")

## 3. Overfitting: train vs test RMSE
Only the training set is used for optimization; the test set is measured (frozen parameters) at each iteration.

Only ratios τ/λ and γ/λ matter: multiplying λ, τ, γ by the same constant leaves the MAP solution unchanged. We fix λ = 1 and vary τ. The run `τ = 10⁴` forces factors ≈ 0 and produces the bias-only baseline.

In [ ]:
ITERS, LAM, GAM = 15, 1.0, 0.1
grid = [(10, 1.), (20, 1.), (10, 10.), (20, 10.), (10, 50.), (20, 50.), (10, 1e4)]
runs = {}
for k, tau in grid:
    print(f"== K={k}  tau={tau:g}")
    runs[(k, tau)] = als(Ru, Ri, Rte, k, LAM, GAM, tau, ITERS, mu, verbose=(tau != 1e4))
    gc.collect()

print(f"\n{'config':>14} {'train':>8} {'test':>8} {'gap':>8} {'best test it.':>14}")
for (k, tau), r in runs.items():
    h = r['hist']
    print(f"K={k:<3} tau={tau:<6g} {h['train'][-1]:8.4f} {h['test'][-1]:8.4f} {h['test'][-1]-h['train'][-1]:8.4f}"
          f" {int(np.argmin(h['test']))+1:>10d} ({min(h['test']):.4f})")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
ref = runs[(10, 1e4)]['hist']
for ax, K in zip(axes, (10, 20)):
    for (k, tau), c in zip([g for g in grid if g[0] == K], plt.cm.tab10.colors):
        h = runs[(k, tau)]['hist']
        ax.plot(range(1, ITERS+1), h['train'], '-',  color=c, label=f'train τ={tau:g}')
        ax.plot(range(1, ITERS+1), h['test'],  '--', color=c, label=f'test τ={tau:g}')
    ax.plot(range(1, ITERS+1), ref['test'], ':', color='k', label='test, bias only')
    ax.set_title(f'K={K}')
    ax.set_xlabel('iteration')
    ax.grid(alpha=.3)
    ax.legend(fontsize=8)
axes[0].set_ylabel('RMSE')
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for (k, tau), r in runs.items():
    axes[0].plot(range(1, ITERS+1), r['hist']['loss'], '-', label=f'K={k}, τ={tau:g}')
axes[0].set_title('Negative log posterior')
axes[0].set_xlabel('iteration')
axes[0].grid(alpha=.3)
axes[0].legend(fontsize=8)

for (k, tau), r in runs.items():
    axes[1].plot(range(1, ITERS+1), r['hist']['train'], '-', label=f'K={k}, τ={tau:g} train')
    axes[1].plot(range(1, ITERS+1), r['hist']['test'], '--', color=axes[1].lines[-1].get_color())
axes[1].set_title('Train (solid) and test (dashed) RMSE')
axes[1].set_xlabel('iteration')
axes[1].grid(alpha=.3)
axes[1].legend(fontsize=7)
plt.tight_layout()
fig.savefig("figures/fig-loss-rmse.png", dpi=300, bbox_inches="tight")
plt.show()

### 3.1 Extended grid: K and λ sweeps
K ∈ {0, 2, 4, 8, 16, 32} with τ = 10, and λ ∈ {0.1, 1, 10} with K = 20, τ = 10.

In [ ]:
ITERS2 = 15

grid_K = [(k, 10.0) for k in (2, 4, 8, 16, 32)]
runs_K = {}
for k, tau in grid_K:
    print(f"== K={k}  tau={tau:g}")
    runs_K[(k, tau)] = als(Ru, Ri, Rte, k, LAM, GAM, tau, ITERS2, mu, verbose=(k in (8, 16, 32)))
    gc.collect()

print("== K=0 (bias only)  tau=1e4")
runs_K[(0, 10.0)] = als(Ru, Ri, Rte, 1, LAM, GAM, 1e4, ITERS2, mu, verbose=False)
gc.collect()

grid_lam = [(20, 10.0, lam) for lam in (0.1, 1.0, 10.0)]
runs_lam = {}
for k, tau, lam in grid_lam:
    print(f"== K={k}  tau={tau:g}  lambda={lam:g}")
    runs_lam[(k, tau, lam)] = als(Ru, Ri, Rte, k, lam, GAM, tau, ITERS2, mu)
    gc.collect()

print(f"\n{'K':>3} {'λ':>4} {'τ':>6} {'train':>8} {'test':>8} {'best test it.':>14}")
for (k, tau), r in sorted(runs_K.items()):
    h = r['hist']
    print(f"{k:3d} {1.0:4g} {tau:6g} {h['train'][-1]:8.4f} {h['test'][-1]:8.4f}"
          f" {int(np.argmin(h['test']))+1:>10d} ({min(h['test']):.4f})")
for (k, tau, lam), r in sorted(runs_lam.items()):
    h = r['hist']
    print(f"{k:3d} {lam:4g} {tau:6g} {h['train'][-1]:8.4f} {h['test'][-1]:8.4f}"
          f" {int(np.argmin(h['test']))+1:>10d} ({min(h['test']):.4f})")

In [ ]:
Ks   = sorted({k for (k, t) in runs_K})
te_K = [min(runs_K[(k, 10.0)]['hist']['test']) for k in Ks]
lams = sorted({lam for (_, _, lam) in runs_lam})
te_l = [min(runs_lam[(20, 10.0, lam)]['hist']['test']) for lam in lams]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(Ks, te_K, 'o-')
axes[0].set_xlabel("latent dimension K")
axes[0].set_ylabel("best test RMSE")
axes[0].set_title("K sweep (τ = 10, λ = 1)")
axes[0].grid(alpha=.3)

axes[1].plot(lams, te_l, 's-', color='C1')
axes[1].set_xscale('log')
axes[1].set_xlabel("λ (with τ = 10, K = 20)")
axes[1].set_ylabel("best test RMSE")
axes[1].set_title("λ sweep (τ/λ varies)")
axes[1].grid(alpha=.3)
plt.tight_layout()
fig.savefig("figures/fig-grid-K-lambda.pdf", bbox_inches="tight")
fig.savefig("figures/fig-grid-K-lambda.png", dpi=300, bbox_inches="tight")
plt.show()

## 4. Overfitting by user activity
Group users by their total number of ratings and compute train / test RMSE per group from the per-user squared errors.

In [ ]:
edges  = [50, 100, 200, 500, 1000]
labels = ["20-49", "50-99", "100-199", "200-499", "500-999", "1000+"]
n_tr, n_te = np.diff(Ru.indptr), np.diff(Rte.indptr)
bins = np.digitize(n_tr + n_te, edges)
nb   = len(labels)
n_users_bin = np.bincount(bins, minlength=nb)

def rmse_by_bin(run):
    tr = np.sqrt(np.bincount(bins, run['sse_tr'], nb) / np.bincount(bins, n_tr, nb))
    te = np.sqrt(np.bincount(bins, run['sse_te'], nb) / np.bincount(bins, n_te, nb))
    return tr, te

requested = [(10, 10.), (20, 10.), (10, 1.), (20, 1.), (10, 1e4)]
show = [k for k in requested if k in runs]
missing = [k for k in requested if k not in runs]
if missing:
    print(f"Missing runs skipped: {missing}")

if show:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    x = np.arange(nb)
    for key, c in zip(show, plt.cm.tab10.colors):
        tr, te = rmse_by_bin(runs[key])
        name = "bias only" if key[1] == 1e4 else f"K={key[0]}, τ={key[1]:g}"
        axes[0].plot(x, tr, '-o', color=c, label=name + ' train')
        axes[0].plot(x, te, '--s', color=c, label=name + ' test')
        axes[1].plot(x, te - tr, '-o', color=c, label=name)
    for ax, t in zip(axes, ("RMSE per user group", "test − train gap (overfitting)")):
        ax.set_xticks(x)
        ax.set_xticklabels([f"{l}\n(n={n:,})" for l, n in zip(labels, n_users_bin)], fontsize=8)
        ax.set_title(t)
        ax.grid(alpha=.3)
        ax.legend(fontsize=7)
    plt.tight_layout()
    fig.savefig("figures/fig-user-groups.png", dpi=300, bbox_inches="tight")
    plt.show()

    print(f"{'group':>9} | " + " | ".join(f"{('K=%d τ=%g' % k) if k[1] != 1e4 else 'bias':>16}" for k in show))
    tabs = {k: rmse_by_bin(runs[k]) for k in show}
    for b in range(nb):
        print(f"{labels[b]:>9} | " + " | ".join(f"{tabs[k][0][b]:.3f} / {tabs[k][1][b]:.3f}  " for k in show))
    print("(format: train / test)")

## 5. Two-dimensional embeddings
Train a K = 2 model with biases and plot the item vectors V. Similar genres should cluster, and opposite genres should lie in opposite regions of the plane.

In [ ]:
K2, TAU2, ITERS_EMB = 2, 10.0, 10
m2 = als(Ru, Ri, Rte, K2, LAM, GAM, TAU2, ITERS_EMB, mu, verbose=False)
V2 = m2['V']

if 'item_cnt' not in globals():
    item_cnt = np.diff(Ru.T.tocsr().indptr)

def find(sub, top=8):
    m = np.where(pd.Series(titles).str.contains(sub, case=False, regex=False).values)[0]
    m = m[np.argsort(-item_cnt[m])][:top]
    return [(int(j), titles[j], int(item_cnt[j])) for j in m]

selection = []
for q in ["Toy Story", "Godfather", "Fellowship of the Ring",
          "Pulp Fiction", "Matrix", "Sleepless in Seattle",
          "Terminator", "Notebook", "Shining", "Amelie",
          "Star Wars", "Titanic", "Alien", "Bambi"]:
    hits = find(q, 1)
    if hits and item_cnt[hits[0][0]] >= 1000:
        selection.append(hits[0][0])

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(V2[:, 0], V2[:, 1], s=6, alpha=0.25, color='gray', label='all movies')
for j in selection:
    ax.scatter(V2[j, 0], V2[j, 1], s=60, color='C3')
    ax.annotate(titles[j][:30], (V2[j, 0], V2[j, 1]),
                fontsize=8, xytext=(4, 4), textcoords='offset points')
ax.set_xlabel("latent trait 1")
ax.set_ylabel("latent trait 2")
ax.grid(alpha=.3)
ax.legend(fontsize=8)
plt.tight_layout()
fig.savefig("figures/fig-embeddings-2d.pdf", bbox_inches="tight")
fig.savefig("figures/fig-embeddings-2d.png", dpi=300, bbox_inches="tight")
plt.show()

print("2D vectors of selected movies:")
for j in selection:
    print(f"  ({V2[j,0]:+.2f}, {V2[j,1]:+.2f})  {titles[j]}")

## 6. Training on all 25M ratings
Select the (K, τ) configuration with the best test RMSE (excluding the bias-only run), then retrain on train + test. The model is saved to Drive.

In [ ]:
best_key = min((k for k in runs if k[1] < 1e3), key=lambda k: min(runs[k]['hist']['test']))
n_it = int(np.argmin(runs[best_key]['hist']['test'])) + 1
print("best configuration:", best_key, "iterations:", n_it)

Ru_full = (Ru + Rte).tocsr()
del Ru, Ri, Rte
gc.collect()
Ri_full = Ru_full.T.tocsr()
mu_full = float(Ru_full.data.mean(dtype=np.float64))
item_cnt = np.diff(Ri_full.indptr)

model = als(Ru_full, Ri_full, None, best_key[0], LAM, GAM, best_key[1], n_it, mu_full)
np.savez(f'{FOLDER}/model_full.npz',
         U=model['U'], V=model['V'], bu=model['bu'], bi=model['bi'], mu=mu_full)

## 7. Synthetic user: do the recommendations make sense?
For a user not in the model, we do not retrain anything: we solve for the user's trait vector with V and item biases fixed (this is exactly the U_m update on that user's ratings). Score: `U·V_n + w·b_n`, with w = 1 then 0.05, with or without a minimum-rating filter.

In [ ]:
def find(sub, top=8):
    m = np.where(pd.Series(titles).str.contains(sub, case=False, regex=False).values)[0]
    m = m[np.argsort(-item_cnt[m])][:top]
    return [(int(j), titles[j], int(item_cnt[j])) for j in m]

def fold_in(liked, model, n_inner=10, use_bias=False):
    V, bi, mu_, lam, gam, tau = model['V'], model['bi'], model['mu'], model['lam'], model['gam'], model['tau']
    idx = np.array(list(liked.keys()))
    r = np.array(list(liked.values()), dtype=float)
    Vs = V[idx]
    A = tau * np.eye(V.shape[1]) + lam * Vs.T @ Vs
    bu_ = 0.0
    for _ in range(n_inner):
        u = np.linalg.solve(A, lam * Vs.T @ (r - mu_ - bu_ - bi[idx]))
        if use_bias:
            bu_ = lam * np.sum(r - mu_ - bi[idx] - Vs @ u) / (lam * len(r) + gam)
    return u, bu_

def recommend(u, model, w_bias=0.05, min_ratings=0, exclude=(), top=10):
    score = model['V'] @ u + w_bias * model['bi']
    score[item_cnt < min_ratings] = -np.inf
    score[list(exclude)] = -np.inf
    top = min(top, int(np.isfinite(score).sum()))
    sel = np.argpartition(-score, top)[:top]
    sel = sel[np.argsort(-score[sel])]
    return [(titles[j], int(item_cnt[j]), round(float(score[j]), 3)) for j in sel]

model['mu'] = mu_full
for s in ["Fellowship of the Ring", "Two Towers", "Return of the King"]:
    print(find(s, 3))

In [ ]:
fellowship = find("Fellowship of the Ring", 1)[0][0]
dummy = {fellowship: 5.0}

u_dummy, _ = fold_in(dummy, model)
for w, mr in [(1.0, 0), (0.05, 0), (0.05, 100), (0.0, 100)]:
    print(f"\n--- item-bias weight = {w}, filter >= {mr} ratings ---")
    for t, c, s in recommend(u_dummy, model, w, mr, exclude=dummy.keys()):
        print(f"  {s:7.3f}  {t}  ({c} ratings)")

## 8. Polarizing movies
A movie whose trait vector v is long produces very different U·v values across users, so knowing whether a user likes it says much about their taste. We compare ‖v‖, the standard deviation of ratings, and the number of ratings.

In [ ]:
norms = np.linalg.norm(model['V'], axis=1)
stds  = row_std(Ri_full.indptr, Ri_full.data)

def show_top(mask, n=15, title=""):
    cand = np.where(mask)[0]
    cand = cand[np.argsort(-norms[cand])][:n]
    print(f"\n{title}\n{'‖v‖':>6} {'std':>5} {'ratings':>7}  title")
    for j in cand:
        print(f"{norms[j]:6.2f} {stds[j]:5.2f} {item_cnt[j]:7d}  {titles[j]}")

show_top(item_cnt >= 1,    title="No filter")
show_top(item_cnt >= 1000, title="Movies with >= 1000 ratings")
print("\nmean movie norm:", norms[item_cnt >= 1000].mean().round(3))

In [ ]:
min_ratings_for_analysis = 1000
popular_mask = item_cnt >= min_ratings_for_analysis
popular_idx  = np.where(popular_mask)[0]

polarizing_movie_idx = popular_idx[np.argsort(-norms[popular_idx])][0]
consensus_movie_idx  = popular_idx[np.argsort( norms[popular_idx])][0]

polarizing_ratings = Ru_full[:, polarizing_movie_idx].data
consensus_ratings  = Ru_full[:, consensus_movie_idx].data

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

axes[0].hist(polarizing_ratings, bins=np.arange(0.5, 5.6, 1.0),
             density=True, alpha=0.7, color='C1', edgecolor='k')
axes[0].set_title(f'Most polarizing: {titles[polarizing_movie_idx]}\n'
                  f'Norm = {norms[polarizing_movie_idx]:.2f}, Std = {stds[polarizing_movie_idx]:.2f}')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Proportion of ratings')

axes[1].hist(consensus_ratings, bins=np.arange(0.5, 5.6, 1.0),
             density=True, alpha=0.7, color='C0', edgecolor='k')
axes[1].set_title(f'Most consensus: {titles[consensus_movie_idx]}\n'
                  f'Norm = {norms[consensus_movie_idx]:.2f}, Std = {stds[consensus_movie_idx]:.2f}')
axes[1].set_xlabel('Rating')

plt.tight_layout()
fig.savefig("figures/fig-polar-hist.png", dpi=300, bbox_inches="tight")
plt.show()

print(f"Polarizing movie : {titles[polarizing_movie_idx]}  "
      f"(norm={norms[polarizing_movie_idx]:.2f}, std={stds[polarizing_movie_idx]:.2f}, "
      f"{item_cnt[polarizing_movie_idx]} ratings)")
print(f"Consensus movie  : {titles[consensus_movie_idx]}  "
      f"(norm={norms[consensus_movie_idx]:.2f}, std={stds[consensus_movie_idx]:.2f}, "
      f"{item_cnt[consensus_movie_idx]} ratings)")

### 8.1 Sanity check: norm consistency across two disjoint user halves
A long norm could be an artifact of overfitting. We split users into two disjoint halves, train the same configuration on each half, and compare the movie norms. If the norms reflect a property of the movies, they should agree across the two models.

In [ ]:
rng = np.random.default_rng(42)
n_users = Ru_full.shape[0]
perm = rng.permutation(n_users)
half1 = np.sort(perm[: n_users // 2])
half2 = np.sort(perm[n_users // 2 :])

def train_on_subset(user_idx):
    Ru_h = Ru_full[user_idx]
    Ri_h = Ru_h.T.tocsr()
    mu_h = float(Ru_h.data.mean(dtype=np.float64))
    return als(Ru_h, Ri_h, None, 20, LAM, GAM, 10.0, 15, mu_h, verbose=False)

print("Training on half 1...")
m1 = train_on_subset(half1)
print("Training on half 2...")
m2 = train_on_subset(half2)

norms1 = np.linalg.norm(m1['V'], axis=1)
norms2 = np.linalg.norm(m2['V'], axis=1)

mask = item_cnt >= 1000
n1, n2 = norms1[mask], norms2[mask]
corr = np.corrcoef(n1, n2)[0, 1]
print(f"Norm correlation over {mask.sum()} movies (>= 1000 ratings): {corr:.3f}")

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(n1, n2, s=10, alpha=0.4, color='C0')
lim = max(n1.max(), n2.max()) * 1.05
ax.plot([0, lim], [0, lim], 'r--', label='identity')
ax.set_xlabel("‖v‖ (model, half 1)")
ax.set_ylabel("‖v‖ (model, half 2)")
ax.set_title(f"Movie norm consistency (corr = {corr:.3f})")
ax.grid(alpha=.3)
ax.legend()
plt.tight_layout()
fig.savefig("figures/fig-norm-consistency.png", dpi=300, bbox_inches="tight")
fig.savefig("figures/fig-norm-consistency.pdf", bbox_inches="tight")
plt.show()